**Requesitos**

In [ ]:
!pip install transformers datasets peft accelerate torch

In [ ]:
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)
import torch

**Carregar Dataset**

In [ ]:
# 1. Carrega o dataset
dataset = load_dataset('json', data_files='dataset_final_divisoes.jsonl')

# 2. Verifica colunas antes da conversão
print("Colunas originais:", dataset["train"].column_names)

# 3. Cria a coluna 'text'
def convert_to_hf_format(example):
    instruction = example.get('Instruction', example.get('instruction', ''))
    output = example.get('Output', example.get('output', ''))
    return {
        "text": f"Instruction: {instruction}\nOutput: {output}"
    }

dataset = dataset.map(convert_to_hf_format)

# 4. Verifica se a coluna foi criada
print("Colunas após conversão:", dataset["train"].column_names)
print("\nPrimeiro exemplo text:")
print(dataset["train"][0]["text"])

# 5. Divide em treino e teste
dataset = dataset["train"].train_test_split(test_size=0.2)
print("\nDataset final:", dataset)

Colunas originais: ['Instruction', 'Output']
Colunas após conversão: ['Instruction', 'Output', 'text']

Primeiro exemplo text:
Instruction: Quais são as partes da Alecrim utilizadas e como elas são preparadas para uso medicinal?
Output: A Alecrim é uma planta herbácea, perene, aromática, amplamente cultivada. As partes utilizadas incluem folhas, flores e outros tecidos. A preparação para uso medicinal envolve a infusão das folhas em água fervida, geralmente com uma colher de sobremesa, e a aplicação topical de folhas secas ou óleo de folhas em condições de cólicas infantis, doenças da pele, candidíase e hemorróidas.

Dataset final: DatasetDict({
    train: Dataset({
        features: ['Instruction', 'Output', 'text'],
        num_rows: 133
    })
    test: Dataset({
        features: ['Instruction', 'Output', 'text'],
        num_rows: 34
    })
})


**Carregar modelo pré-treinado e tokenizador**

In [ ]:
model_name = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    trust_remote_code=True,
    torch_dtype=torch.float32   # float32 para CPU; use bfloat16 se tiver GPU
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Modelo carregado: {model_name}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Modelo carregado: Qwen/Qwen3-0.6B


**Inferência ANTES do Fine-Tuning**

In [ ]:
def generate_response(model, tokenizer, instruction, input_text=""):
    if input_text:
        prompt = f"Instruction: {instruction}\nInput: {input_text}\nOutput:"
    else:
        prompt = f"Instruction: {instruction}\nOutput:"

    inputs = tokenizer(prompt, return_tensors="pt")

    with torch.no_grad():           # ← adicionar isso, economiza memória
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=50,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=True,
            temperature=0.7
        )

    # Remove os tokens do prompt da saída
    prompt_len = inputs["input_ids"].shape[1]
    generated_ids = outputs[0][prompt_len:]          # ← mais limpo para Qwen3
    resposta = tokenizer.decode(generated_ids, skip_special_tokens=True)
    return resposta.strip()

# Exemplo de instrução (deve existir no dataset.jsonl)

test_instruction = "Quais são os usos e formas de preparo do Alecrim?"

print("=== ANTES DO FINE-TUNING ===")
print(f"Instrução: {test_instruction}")
print(f"Resposta base: {generate_response(base_model, tokenizer, test_instruction)}")

=== ANTES DO FINE-TUNING ===
Instrução: Quais são os usos e formas de preparo do Alecrim?
Resposta base: **Uso do Alecrim**

O Alecrim é um tipo de medicamento comum, geralmente preparado com a combinação de ingredientes naturais e medicamentos, que são usados para tratar diversos tipos de


**Tokenização**

In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

tokenized_datasets = dataset.map(tokenize_function, batched=True)
print("Dataset tokenizado:", tokenized_datasets)

Map:   0%|          | 0/133 [00:00<?, ? examples/s]

Map:   0%|          | 0/34 [00:00<?, ? examples/s]

Dataset tokenizado: DatasetDict({
    train: Dataset({
        features: ['Instruction', 'Output', 'text', 'input_ids', 'attention_mask'],
        num_rows: 133
    })
    test: Dataset({
        features: ['Instruction', 'Output', 'text', 'input_ids', 'attention_mask'],
        num_rows: 34
    })
})


**Configuração LORA**

In [ ]:
# A partir daqui, usaremos uma cópia do modelo base para o fine-tuning
model = base_model  # poderia também ser uma nova instância
model = prepare_model_for_kbit_training(model)

In [ ]:
# Célula auxiliar — verificar os nomes das camadas
for name, module in base_model.named_modules():
    print(name)


model
model.embed_tokens
model.layers
model.layers.0
model.layers.0.self_attn
model.layers.0.self_attn.q_proj
model.layers.0.self_attn.k_proj
model.layers.0.self_attn.v_proj
model.layers.0.self_attn.o_proj
model.layers.0.self_attn.q_norm
model.layers.0.self_attn.k_norm
model.layers.0.mlp
model.layers.0.mlp.gate_proj
model.layers.0.mlp.up_proj
model.layers.0.mlp.down_proj
model.layers.0.mlp.act_fn
model.layers.0.input_layernorm
model.layers.0.post_attention_layernorm
model.layers.1
model.layers.1.self_attn
model.layers.1.self_attn.q_proj
model.layers.1.self_attn.k_proj
model.layers.1.self_attn.v_proj
model.layers.1.self_attn.o_proj
model.layers.1.self_attn.q_norm
model.layers.1.self_attn.k_norm
model.layers.1.mlp
model.layers.1.mlp.gate_proj
model.layers.1.mlp.up_proj
model.layers.1.mlp.down_proj
model.layers.1.mlp.act_fn
model.layers.1.input_layernorm
model.layers.1.post_attention_layernorm
model.layers.2
model.layers.2.self_attn
model.layers.2.self_attn.q_proj
model.layers.2.self_att

In [ ]:
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    !pip install --upgrade torchao>=0.16.0

lora_config = LoraConfig(
    r=16,                    # rank da decomposição
    lora_alpha=32,           # escala alpha
    target_modules=[         # camadas alvo
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    inference_mode=False     # False = modo treinamento
)

model = prepare_model_for_kbit_training(base_model)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 4,587,520 || all params: 600,637,440 || trainable%: 0.7638


**Data Collator**

In [ ]:
# Configura o data collator para linguagem causal
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

**Argumentos de treinamento**

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",          # diretório de saída
    eval_strategy="steps",
    eval_steps=100,
    learning_rate=1e-3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=100,
    weight_decay=0.01,
    logging_steps=10,
    save_strategy="steps",
    save_steps=100,
    load_best_model_at_end=True,
    report_to="none",               # desabilita logging para wandb/mlflow
)

**Iniciar o trainer**

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
)

**Treinar modelo**

In [ ]:
trainer.train()

Step,Training Loss,Validation Loss


**Salvar modelo**

In [22]:
model.save_pretrained("lora_causal_model_1/")
tokenizer.save_pretrained("Qwen/Qwen3-0.6B_tokenizer")

('Qwen/Qwen3-0.6B_tokenizer/tokenizer_config.json',
 'Qwen/Qwen3-0.6B_tokenizer/chat_template.jinja',
 'Qwen/Qwen3-0.6B_tokenizer/tokenizer.json')

**Inferência APÓS o Fine-Tuning**

In [23]:
# Carrega o modelo fine-tunado (apenas os adaptadores LoRA)
finetuned_model = AutoModelForCausalLM.from_pretrained("lora_causal_model_1/")
finetuned_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B_tokenizer")

# Garante que o token de padding esteja definido
if finetuned_tokenizer.pad_token is None:
    finetuned_tokenizer.pad_token = finetuned_tokenizer.eos_token

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/224 [00:00<?, ?it/s]

In [24]:
print("=== DEPOIS DO FINE-TUNING ===")
print(f"Instrução: {test_instruction}")
print(f"Resposta ajustada: {generate_response(finetuned_model, finetuned_tokenizer, test_instruction)}")

=== DEPOIS DO FINE-TUNING ===
Instrução: Quais são os usos e formas de preparo do Alecrim?
Resposta ajustada: O Alecrim é utilizado para uso tópico local para aliviar sintomas de problemas digestivos (como flatemãos e dores de cabeça), para uso na medicina tradicional, e também é utilizada como uma
